# Share Variability Across Partitions

Standard deviation $\sigma$ (percentage points) of each level's share across the $K=20$ partitions, for the feature-hierarchical and k-means partitioners.

## Setup

In [1]:
from pathlib import Path
import sys, warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA
from sklearn.preprocessing import QuantileTransformer, StandardScaler

HERE = Path.cwd()
if str(HERE) not in sys.path:
    sys.path.insert(0, str(HERE))

from partitioning import _build_partitioner

warnings.filterwarnings('ignore')
np.random.seed(42)

METADATA_CSV = HERE / 'files' / 'train_N28.csv'
N_PARTITIONS = 20
SEED = 42
RANK_UTILIZATION = {
    'type': 'rank_utilization',
    'column': 'utilization',
    'keep_in_output': False,
    'levels': {'low': [0.70, 0.75], 'medium': [0.80], 'high': [0.85, 0.90]},
}

STRATEGIES = [
    {'name': 'feature_hierarchical', 'cfg': {'type': 'feature_hierarchical', 'n_partitions': N_PARTITIONS, 'features': ['design_name', 'utilization', 'clock'], 'preprocessing': [RANK_UTILIZATION], 'cluster_share': 0.8, 'dirichlet_alpha': 0.5, 'random_state': SEED}},
    {'name': 'kmeans', 'cfg': {'type': 'kmeans', 'n_partitions': N_PARTITIONS, 'cluster_share': 0.8, 'dirichlet_alpha': 0.5, 'random_state': SEED}},
]


## Load Metadata

In [2]:
df = pd.read_csv(METADATA_CSV)
META_COLS = [c for c in df.columns if c not in ('filename', 'sample_id')]

## Run Partitioners

In [3]:
def build_assignment(parts):
    assignment = np.full(len(df), -1, dtype=int)
    lookup = {name: i for i, name in enumerate(df['filename'].to_numpy())}
    for i, p in enumerate(parts):
        for name in p['filename']:
            assignment[lookup[name]] = i
    return assignment


results = {}
for entry in STRATEGIES:
    part = _build_partitioner(dict(entry['cfg']))
    parts = part.partition(df)
    results[entry['name']] = {
        'partitioner': part,
        'parts': parts,
        'assignment': build_assignment(parts),
        'sizes': [len(p) for p in parts],
    }

STRAT_NAMES = list(results)

## Table

For each factor level, its percentage share within each partition is computed, then the standard deviation of that share over the 20 partitions (population $\sigma$, `ddof=0`). *mean σ* averages the level σ's of a factor.

In [4]:
def factor_levels(series):
    """Level labels of a factor, numerically ordered when possible."""
    vals = series.astype(str).unique()
    try:
        return sorted(vals, key=float)
    except ValueError:
        return sorted(vals)


def share_sigma(frame, labels, factor):
    """Std (pp) across partitions of each level's % share within the partition."""
    counts = pd.crosstab(pd.Series(labels, index=frame.index, name='partition'),
                         frame[factor].astype(str))
    counts = counts.reindex(columns=factor_levels(frame[factor]), fill_value=0)
    pct = counts.div(counts.sum(axis=1), axis=0) * 100
    return pct.std(axis=0, ddof=0)


FACTORS = ['design_name', 'clock', 'utilization']
COLUMNS = {'feature_hierarchical': 'Feature-hierarchical', 'kmeans': 'K-means'}

rows = []
for f in FACTORS:
    sig = pd.DataFrame({title: share_sigma(df, results[n]['assignment'], f)
                        for n, title in COLUMNS.items()})
    sig.loc['mean σ'] = sig.mean()
    sig.index = pd.MultiIndex.from_product([[f], sig.index], names=['Factor', 'Level'])
    rows.append(sig)

sigma_table = pd.concat(rows)
display(sigma_table.style.format('{:.2f}'))